In [ ]:
import os
import pandas as pd

movie_title_path = os.path.join("..", "data", "movie_titles.csv")
train_data_path = os.path.join("..", "final_data", "Train_Data.csv")

# Load Movie Metadata for mapping
movies_df = pd.read_csv(movie_title_path, encoding="latin1", names=["Movie_ID", "Year", "Title"], header=None, on_bad_lines="skip")
movie_id_to_title = movies_df.set_index("Movie_ID")["Title"].to_dict()

df_train = pd.read_csv(train_data_path)

#### Using the svd predictions generated from the prev notebook

In [26]:
import pickle
import os

path = os.path.join("..", "final_data", "svd_predictions.pkl")
with open(path, "rb") as f:
    svd_predictions = pickle.load(f)

model_path = os.path.join("..", "final_data", "svd_model.pkl")
with open(model_path, "rb") as f:
    svd = pickle.load(f)

print(f"Loaded {len(svd_predictions):,} SVD predictions.")

Loaded 376,590 SVD predictions.


In [ ]:
import pandas as pd
import numpy as np
import os
from collections import defaultdict

# loading the movie titles metadata
try:
    movies_df = pd.read_csv(
        movie_title_path,
        encoding="latin1",
        names=["Movie_ID", "Year", "Title"],
        header=None,
        on_bad_lines="skip"  # Prevent crashes from row structural errors
    )
except Exception as e:
    print(f"Encoding fallback triggered due to: {e}")
    movies_df = pd.read_csv(
        movie_title_path,
        encoding="utf-8",
        names=["Movie_ID", "Year", "Title"],
        header=None,
        on_bad_lines="skip"
    )

movie_id_to_title = movies_df.set_index("Movie_ID")["Title"].to_dict()
print(f"Mapped {len(movie_id_to_title):,} movies to their titles.")

# Reconstruct test profiles from svd predictions
user_test_profiles = defaultdict(list)
for u_id, m_id, true_r, est, _ in svd_predictions:
    user_test_profiles[u_id].append((m_id, true_r, est))

# Isolate high density users (>= 10 test items) same cohort used for MAP@10
dense_target_users = [
    u_id for u_id, items in user_test_profiles.items() if len(items) >= 10
]

print(f"Isolated {len(dense_target_users):,} high-density users for Top-10 generation.")

Mapped 17,434 movies to their titles.
Isolated 2,648 high-density users for Top-10 generation.


In [28]:
import pandas as pd

print("PHASE 2: TOP-10 RECOMMENDATION GENERATION")
print("=" * 85)

# Pick the 3 most active users from the test profiles
sample_users = sorted(dense_target_users, key=lambda u: len(user_test_profiles[u]), reverse=True)[:3]

for idx, user_id in enumerate(sample_users, 1):
    print(f"\n{'=' * 85}")
    print(f"USER RECOMMENDATION REPORT #{idx}  (User ID: {user_id})")
    print(f"{'=' * 85}")

    user_data = user_test_profiles[user_id]
    profile_list = []

    for m_id, true_r, est in user_data:
        title = movie_id_to_title.get(m_id, f"Unknown Movie (ID: {m_id})")
        profile_list.append({
            "Movie Title": title,
            "Actual Rating": int(true_r),
            "SVD Predicted": round(est, 2),
            "Absolute Error": round(abs(true_r - est), 2)
        })

    df_user = pd.DataFrame(profile_list)
    
    # Sort by SVD Predicted to look at the model's highest predictions in the test set
    df_top10 = df_user.sort_values(by="SVD Predicted", ascending=False).head(10)

    print(f"Top-10 Model Predictions (From Test Split Evaluation):")
    print("-" * 85)
    
    # Using formatters keeps column layout cleanly aligned regardless of long titles
    print(df_top10.to_string(
        index=False, 
        justify="left",
        formatters={
            "Movie Title": lambda x: f"{x[:45]:<45}", # Clip extra long titles cleanly
            "Actual Rating": lambda x: f"{x:^16}",
            "SVD Predicted": lambda x: f"{x:^15.2f}",
            "Absolute Error": lambda x: f"{x:^15.2f}"
        }
    ))
    print("=" * 85)

PHASE 2: TOP-10 RECOMMENDATION GENERATION

USER RECOMMENDATION REPORT #1  (User ID: 305344)
Top-10 Model Predictions (From Test Split Evaluation):
-------------------------------------------------------------------------------------
Movie Title                                   Actual Rating    SVD Predicted   Absolute Error 
Farscape: Season 4                                   1              3.63            2.63      
Mr. and Mrs. Iyer                                    1              3.54            2.54      
The Best Bits of Mr. Bean                            2              3.36            1.36      
Southern Comfort                                     3              3.23            0.23      
The Sorrow and the Pity                              1              3.18            2.18      
Trailer Park Boys: Season 4                          1              3.18            2.18      
Dolemite                                             2              3.15            1.15      
Queen: 

In [ ]:
print("PHASE 2.5: GENERATING TRUE DISCOVERY RECOMMENDATIONS (UNSEEN ITEMS)")
print("=" * 85)

# 1. Get all unique movie IDs available in your metadata catalog
all_movie_ids = set(movies_df["Movie_ID"].unique())

for idx, user_id in enumerate(sample_users, 1):
    print(f"\n{'=' * 85}")
    print(f"TRUE TOP-10 RECS FOR DISCOVERY #{idx}  (User ID: {user_id})")
    print(f"{'=' * 85}")
    
    # Find items this user has already interacted with during training
    seen_movies = set(df_train[df_train["User_ID"] == user_id]["Movie_ID"])
    
    # Filter down to strictly unseen items
    unseen_movies = all_movie_ids - seen_movies
    
    discovery_list = []
    
    for m_id in unseen_movies:
        # LIVE INFERENCE: Predict the score for this unseen movie
        pred = svd.predict(user_id, m_id)
        
        title = movie_id_to_title.get(m_id, f"Unknown Movie (ID: {m_id})")
        discovery_list.append({
            "Movie Title": title,
            "SVD Predicted Rating": round(pred.est, 2)
        })
        
    df_discovery = pd.DataFrame(discovery_list)
    
    # Sort and get top 10 highest predicted ratings for the user
    df_top10_unseen = df_discovery.sort_values(by="SVD Predicted Rating", ascending=False).head(10)
    
    print(df_top10_unseen.to_string(
        index=False,
        justify="left",
        formatters={
            "Movie Title": lambda x: f"{str(x)[:50]:<50}",
            "SVD Predicted Rating": lambda x: f"{x:^25.2f}"
        }
    ))
    print("=" * 85)

PHASE 2.5: GENERATING TRUE DISCOVERY RECOMMENDATIONS (UNSEEN ITEMS)

TRUE TOP-10 RECS FOR DISCOVERY #1  (User ID: 305344)
Movie Title                                        SVD Predicted Rating     
24: Season 1                                                 4.50           
Dead Like Me: Season 2                                       4.45           
Beauty and the Beast: Special Edition                        4.42           
Monty Python's Flying Circus                                 4.36           
Horatio Hornblower                                           4.32           
To Kill a Mockingbird                                        4.30           
Carnivale: Season 1                                          4.12           
The Visitors                                                 4.07           
The Royal Tenenbaums                                         4.07           
Alias: Season 3                                              4.05           

TRUE TOP-10 RECS FOR DISCOVERY